In [3]:
import ollama
ollama.embeddings(model='nomic-embed-text', prompt='The sky is blue because of rayleigh scattering')

EmbeddingsResponse(embedding=[0.5891852378845215, 0.40087997913360596, -3.303852081298828, -0.5256258249282837, 0.7496368885040283, 1.5192068815231323, -0.12541252374649048, 0.39620473980903625, 0.0680127739906311, -1.1085814237594604, 0.6923737525939941, 1.2783215045928955, 1.1457270383834839, 1.088584303855896, 0.25080519914627075, 0.29336944222450256, 0.15155696868896484, -0.634049117565155, -0.20992401242256165, -0.19607779383659363, -1.7958388328552246, -0.6294044256210327, 0.03828483074903488, -0.6683821678161621, 1.2616585493087769, 1.2775959968566895, -0.15969999134540558, -0.002039968967437744, -0.2970771789550781, -0.4805387258529663, 1.2056541442871094, -0.6378178596496582, -0.5401647090911865, -1.0358539819717407, 0.6314536929130554, -1.209057331085205, 0.6832631230354309, -0.05838744342327118, -0.19721558690071106, 0.127748042345047, -0.014416977763175964, -0.554468035697937, 0.35206887125968933, 0.045384641736745834, 0.5979752540588379, -0.9549010396003723, 0.507586836814

In [10]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
import asyncio
import os

# Load data
documents = SimpleDirectoryReader("erav3_content\\course_notes").load_data()
# load more documents from a directory
documents += SimpleDirectoryReader("erav3_content\\course_transcripts").load_data()

print(f"Loaded {len(documents)} documents")

Loaded 709 documents


In [19]:
documents[0]

Document(id_='d663cad3-4c97-42e9-9fb8-d76f35d22a9b', embedding=None, metadata={'page_label': '1', 'file_name': 's1.pdf', 'file_path': 'h:\\My_studies\\ERAv3\\Assignment - 26\\erav3_content\\course_notes\\s1.pdf', 'file_type': 'application/pdf', 'file_size': 6138106, 'creation_date': '2025-05-02', 'last_modified_date': '2025-05-02'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='Session 1: Introduction to AI, Neural Networks, and\nDevelopment Tools\nDue Oct 19, 2024 by 11:59pm\nPoints None\nAvailable after Oct 12, 2024 at 11am\nSession 1: Introductionto AI, Neural Networks,and Development Tools\n \n \n- Course Structure and Expectations: 

In [25]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# initialize client, setting path to save data
db = chromadb.PersistentClient(path="./chroma_db")

# create collection
chroma_collection = db.get_or_create_collection("erav3")

# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [26]:
embed_model = OllamaEmbedding(
    model_name="nomic-embed-text",
    base_url="http://localhost:11434",
    request_timeout=360.0)

llm = Ollama(model="llama3.1:8b", request_timeout=360.0)

In [ ]:
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
    embed_model=embed_model,
)
query_engine = index.as_query_engine(llm=llm, similarity_top_k=3)

In [28]:
# define tools
def multiply(a: float, b: float) -> float:
    """Useful for multiplying two numbers."""
    return a * b


async def search_documents(query: str) -> str:
    """Useful for answering natural language questions about an personal essay written by Paul Graham."""
    response = await query_engine.aquery(query)
    print(f"Query: {query}")
    print(f"Response: {response}")
    return str(response)

In [ ]:
# Create an enhanced workflow with both tools
agent = AgentWorkflow.from_tools_or_functions(
            [multiply, search_documents],
            llm=llm,
            system_prompt="""You are a helpful assistant that can perform calculations
            and search through documents to answer questions.
            
            Only use the calculator when the user asks for a calculation.
            Otherwise, STRICTLY use search_documents to search through the documents to answer the question.
            """,
            
)


In [30]:
# Now we can ask questions about the documents or do calculations
async def main():
    response = await agent.run(
        "Use search tool and say what is MCP "
    )
    print(response)


# Run the agent
if __name__ == "__main__":
    asyncio.run(main())

Query: MCP
Response: A protocol that enables models to execute tasks directly by following clear instructions. It's a significant advancement over traditional LLMs, which often relied on statistical likelihood and inferred context in unintended ways, leading to suboptimal results. By standardizing resources, prompts, and tools, MCP allows for more precise control over model behavior and enables the creation of truly intelligent and autonomous AI ecosystems.
Tool called search_documents returned a response that can be used to answer the user's question.

The user asked 'what is MCP'. The search tool's response was:
"A protocol that enables models to execute tasks directly by following clear instructions." 
So, the response to the original question would be: MCP is a protocol that enables models to execute tasks directly by following clear instructions.


In [32]:
await agent.run("What is depthwise separable convolution?")

Query: What is depthwise separable convolution?
Response: Depthwise separable convolution is a technique that splits a regular convolution into two smaller convolutions. It involves first performing a depthwise convolution where each filter only slides over one channel of the input feature maps, and then performing a pointwise convolution (equivalent to a 1x1 convolution) on the result to combine the channels.


AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'tool_calls': []}, blocks=[TextBlock(block_type='text', text='Alternatively, you can also view it as a combination of two separate operations: Depthwise Convolution and Pointwise Convolution.\n\nIn Depthwise Convolution, each filter only slides over one channel of the input feature maps.\n\nThe output is then passed through a Pointwise Convolution (which is equivalent to 1x1 convolution), where each element in the output from the depthwise convolution is multiplied with every other element in that same spatial position')]), tool_calls=[ToolCallResult(tool_name='search_documents', tool_kwargs={'query': 'What is depthwise separable convolution?'}, tool_id='search_documents', tool_output=ToolOutput(content='Depthwise separable convolution is a technique that splits a regular convolution into two smaller convolutions. It involves first performing a depthwise convolution where each filter only sli

In [35]:
print(await agent.run("What is ERA?"))

Query: ERA
Response: Vision
The search results indicate that ERA refers to Earned Run Average, which is a statistical measure in baseball that represents the average number of earned runs a pitcher allows per nine innings pitched.


In [ ]:
import chromadb
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# initialize client
db = chromadb.PersistentClient(path="./chroma_db")

# get collection
chroma_collection = db.get_or_create_collection("erav3")

# assign chroma as the vector_store to the context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)


index = VectorStoreIndex.from_vector_store(
    vector_store, 
    storage_context=storage_context,
    embed_model=embed_model,
)
query_engine = index.as_query_engine(similarity_top_k=3)

ERA appears to be an acronym for a course or educational program. It seems to be structured around various topics and concepts related to computer science and programming, including NLP/Audio, Vision vs NLP, and theoretical explanations for emergence in complex systems. The context also mentions videos and transcripts from "STUDIO (ERA V3 Session - 9 - Transcript)", which suggests that ERA is likely an online course with multimedia content.


In [39]:
response = query_engine.query("expand ERA and give full form?")
print(response)

ERA stands for Emergent Reality Abstraction or possibly other abbreviations depending on context. However, based on the given text it seems more likely to be related to "Emergence" which is a concept in various fields such as physics, computer science, and philosophy.


In [70]:
retriever = index.as_retriever(similarity_top_k=3)
results = retriever.retrieve("What is depthwise separable convolution?")
results

[NodeWithScore(node=TextNode(id_='654c1ec3-6013-4765-8399-2820107d8364', embedding=None, metadata={'page_label': '38', 'file_name': 's8.pdf', 'file_path': 'h:\\My_studies\\ERAv3\\Assignment - 26\\erav3_content\\course_notes\\s8.pdf', 'file_type': 'application/pdf', 'file_size': 9790446, 'creation_date': '2025-05-02', 'last_modified_date': '2025-05-02'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='17751e84-4c79-494f-9a7d-05c34aa6b1e0', node_type='4', metadata={'page_label': '38', 'file_name': 's8.pdf', 'file_path': 'h:\\My_studies\\ERAv3\\Assignment - 26\\erav3_content\\course_notes\\s8.pdf', 'file_type': 'application/pdf', 'file_size': 9790446, 'creation_date': '2025-05-02', 'last_modified_date': '2025-

In [74]:
print(results[0].node.get_content())
print(f"File path: {results[0].node.metadata['file_path']}")
print(f"Page label: {results[0].node.metadata['page_label']}")

For a depthwise separable convolution on the same example, we traverse the 16 channels with 1 3x3
kernel each, giving us 16 feature maps.
Now, before merging anything, we traverse these 16 feature maps with 32 1x1 convolutions each and
only then start to them add together.
This results in 656 (16x3x3 + 16x32x1x1) parameters as opposed to the 4608 (16x32x3x3)
parameters from above. [REF  (https://eli.thegreenplace.net/2018/depthwise-separable-convolutions-
for-machine-learning) ]
 
 
 
Spatially Separable Convolutions
 
 
 
Was used immensely in different variants of Xception-Inception Networks as well as in MobileNets.\
 
 

File path: h:\My_studies\ERAv3\Assignment - 26\erav3_content\course_notes\s8.pdf
Page label: 38


In [ ]:
f"file:///{str(results[0].node.metadata['file_path']).replace(' ', '%20')}#page={results[0].node.metadata['page_label']}"

'file:///h:\\My_studies\\ERAv3\\Assignment%20-%2026\\erav3_content\\course_notes\\s8.pdf#page=38'